<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 1 (Lunes) — Predicción — Series de tiempo**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🌡️ Series de tiempo (SARIMA): temperatura de sensores IoT

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Sensores IoT registran temperatura dentro y fuera de una sala; queremos pronosticar las próximas horas. |
| 🔍 Comprensión de los datos | Cargamos `IOT-temp.csv`, revisamos fechas, tipos y faltantes. |
| 🧹 Preparación de los datos | Convertimos fechas, eliminamos columnas irrelevantes y agregamos la temperatura **promedio por hora**. |
| 🤖 Modelado | Descomposición estacional y búsqueda de parámetros SARIMA con el menor AIC. |
| 📊 Evaluación | Comparamos predicción vs. valores reales (MSE y RMSE) y revisamos diagnósticos. |
| 🚀 Despliegue y conclusiones | Pronosticamos las próximas 100 horas con intervalos de confianza. |

> 🎯 **Analogía:** un termómetro inteligente que anota la temperatura cada hora: con su historial, SARIMA aprende los ciclos del día y anticipa las próximas horas.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 👉 **1** | Lunes | **Predicción: regresión lineal y series de tiempo** |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('IOT-temp.csv', 'dia_2')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

 # Algoritmos de predicción

Por serie de tiempo nos referimos a datos estadísticos que se recopilan, observan o registran en intervalos de tiempo regulares (diario, semanal, semestral, anual, entre otros). El término serie de tiempo se aplica por ejemplo a datos registrados en forma periódica que muestran, por ejemplo, las ventas anuales totales de almacenes, el valor trimestral total de contratos
de construcción otorgados, el valor trimestral del PIB.

---
## 🎯 Fase 1 · Comprensión del negocio

Sensores IoT registran temperatura dentro y fuera de una sala; queremos pronosticar las próximas horas.

📚 Lectura recomendada: [3]

## Análisis del Problema

Internet ha evolucionado rápidamente y esto ha permitido que IoT sea ya una realidad y no sólo una visión de futuro. La fama de esta tecnología radica principalmente en todas las aplicaciones y posibilidades que nos proporciona tanto para mejorar tanto la vida cotidiana de las personas como los entornos empresariales, dónde ya se está implantando desde hace algún tiempo.


La forma más fácil de comenzar es conectar un sensor analógico al Arduino. Un sensor analógico,que proporciona una señal continua.

### 📦 Librerías

In [ ]:
import warnings
import itertools
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
plt.style.use('fivethirtyeight')
import pandas as pd
import statsmodels.api as sm
import matplotlib
matplotlib.rcParams['axes.labelsize'] = 14
matplotlib.rcParams['xtick.labelsize'] = 12
matplotlib.rcParams['ytick.labelsize'] = 12
matplotlib.rcParams['text.color'] = 'k'

---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos `IOT-temp.csv`, revisamos fechas, tipos y faltantes.

📚 Lectura recomendada: [7]

### 📥 Carga de datos

In [ ]:
data = pd.read_csv('sample_data/IOT-temp.csv')

In [ ]:
data.columns

In [ ]:
data["noted_date"]=pd.to_datetime(data["noted_date"],errors='coerce')
data["hour"]=data["noted_date"].dt.hour
data["temp"]=pd.to_numeric(data["temp"], downcast='float')

In [ ]:
#shape
data.shape

In [ ]:
#head of data
data.head()

#tail of data
data.tail()

In [ ]:
#now lets check shape
data.shape

#lets verify whether nan is present in columns
'nan' in data.columns
#so its proved that nan is not in columns

In [ ]:
# Numero de registros del dataset
print("Cantidad de registros:",len(data))
# Cantidad de columnas del dataset
print("Cantidad de columnas:",len(data.columns))
# Tipos de datos del dataset
print("Tipo de datos:",data.dtypes.unique())
# Tamaño del daset
print("Filas y columnas:",data.shape)

In [ ]:
data.head()

In [ ]:
#Información del dataset
data.info()

---
## 🧹 Fase 3 · Preparación de los datos

Convertimos fechas, eliminamos columnas irrelevantes y agregamos la temperatura **promedio por hora**.

📚 Lectura recomendada: [4]

In [ ]:
cols = ['id', 'room_id/id','out/in']
data.drop(cols, axis=1, inplace=True)

In [ ]:
data = data.sort_values('noted_date')
data.isnull().sum()

In [ ]:
data = data.groupby('noted_date')['temp'].mean().reset_index()

In [ ]:
data = data.set_index('noted_date')
data.index

In [ ]:
data

---
## 🔍 Fase 2 · Comprensión de los datos — patrón horario y descomposición

Cargamos `IOT-temp.csv`, revisamos fechas, tipos y faltantes.

#### Temperatura promedio por hora

In [ ]:
y = data['temp'].resample(rule='1h').mean().interpolate()

In [ ]:
y

In [ ]:
#y = data['temp'].resample('W').mean()
y.plot(figsize=(15, 6))
plt.show()

In [ ]:
from pylab import rcParams
rcParams['figure.figsize'] = 18, 8
decomposition = sm.tsa.seasonal_decompose(y, model='additive')
fig = decomposition.plot()
plt.show()

---
## 🤖 Fase 4 · Modelado

Descomposición estacional y búsqueda de parámetros SARIMA con el menor AIC.

📚 Lectura recomendada: [5] [6]

Los modelos ARIMA se denotan con la notación ARIMA (p, d, q). Estos tres parámetros tienen en cuenta la estacionalidad, la tendencia y el ruido en los datos:

In [ ]:
p = d = q = range(0, 2)
pdq = list(itertools.product(p, d, q))
seasonal_pdq = [(x[0], x[1], x[2], 12) for x in list(itertools.product(p, d, q))]
print('Examples of parameter combinations for Seasonal ARIMA...')
print('SARIMAX: {} x {}'.format(pdq[1], seasonal_pdq[1]))
print('SARIMAX: {} x {}'.format(pdq[1], seasonal_pdq[2]))
print('SARIMAX: {} x {}'.format(pdq[2], seasonal_pdq[3]))
print('SARIMAX: {} x {}'.format(pdq[2], seasonal_pdq[4]))

In [ ]:
for param in pdq:
    for param_seasonal in seasonal_pdq:
        try:
            mod = sm.tsa.statespace.SARIMAX(y,
                                            order=param,
                                            seasonal_order=param_seasonal,
                                            enforce_stationarity=False,
                                            enforce_invertibility=False)
            results = mod.fit()
            print('ARIMA{}x{}12 - AIC:{}'.format(param, param_seasonal, results.aic))
        except:
            continue

In [ ]:
mod = sm.tsa.statespace.SARIMAX(y,
                                order=(1, 1, 1),
                                seasonal_order=(1, 1, 0, 12),
                                enforce_stationarity=False,
                                enforce_invertibility=False)
results = mod.fit()
print(results.summary().tables[1])

---
## 📊 Fase 5 · Evaluación

Comparamos predicción vs. valores reales (MSE y RMSE) y revisamos diagnósticos.

📚 Lectura recomendada: [4]

> ⚠️ **Nota:** parte del texto interpretativo de este cuaderno proviene de una versión en R; al ejecutar en Python los números exactos (AIC, coeficientes, RMSE) pueden variar ligeramente. Use las salidas de sus propias celdas para interpretar.

In [ ]:
pred = results.get_prediction(start=pd.to_datetime('2018-01-11 00:00:00'), dynamic=False)
pred_ci = pred.conf_int()
ax = y['2018-01-11 00:00:00':].plot(label='observed')
pred.predicted_mean.plot(ax=ax, label='One-step ahead Forecast', alpha=.7, figsize=(14, 7))
ax.fill_between(pred_ci.index,
                pred_ci.iloc[:, 0],
                pred_ci.iloc[:, 1], color='k', alpha=.2)
ax.set_xlabel('Date')
ax.set_ylabel('Temp')
plt.legend()
plt.show()

In [ ]:
y_forecasted = pred.predicted_mean
y_truth = y['2018-12-01 01:00:00':]
mse = ((y_forecasted - y_truth) ** 2).mean()
print('The Mean Squared Error of our forecasts is {}'.format(round(mse, 2)))

In [ ]:
print('The Root Mean Squared Error of our forecasts is {}'.format(round(np.sqrt(mse), 2)))

---
## 🚀 Fase 6 · Despliegue y conclusiones

Pronosticamos las próximas 100 horas con intervalos de confianza.

📚 Lectura recomendada: [4]

In [ ]:
pred_uc = results.get_forecast(steps=100)
pred_ci = pred_uc.conf_int()
ax = y.plot(label='observed', figsize=(14, 7))
pred_uc.predicted_mean.plot(ax=ax, label='Forecast')
ax.fill_between(pred_ci.index,
                pred_ci.iloc[:, 0],
                pred_ci.iloc[:, 1], color='k', alpha=.25)
ax.set_xlabel('Date')
ax.set_ylabel('Temp')
plt.legend()
plt.show()

In [ ]:
pred_ci

---
## 🧪 Refuerzo: línea base: ¿SARIMA le gana a un pronóstico ingenuo?

Antes de celebrar un modelo, hay que compararlo con una **línea base** simple [4]: el pronóstico **ingenuo estacional** repite el valor de la misma hora del día anterior (24 horas atrás). Si SARIMA no lo supera, la complejidad no se justifica.

In [ ]:
import numpy as np
h = 100                                  # horas a predecir
train, test = y.iloc[:-h], y.iloc[-h:]
ingenuo = y.shift(24).iloc[-h:]          # mismo momento del día anterior
rmse = lambda a, b: float(np.sqrt(np.mean((a.values - b.values) ** 2)))
print(f'RMSE ingenuo estacional (24 h): {rmse(test, ingenuo):.2f}')
print('Compare con el RMSE de SARIMA calculado arriba.')

### ✍️ Ejercicios
1. Cambie el rezago a 1 hora (ingenuo simple): ¿mejora o empeora?
2. ¿Cuántas horas de datos necesita SARIMA para aprender un ciclo diario?

---
## ✅ Conclusiones y siguientes pasos

La mejora clave frente a la versión anterior fue **promediar** (no sumar) las lecturas de todos los sensores: sumar temperaturas de sensores distintos no tiene sentido físico. Con datos horarios, SARIMA captura el ciclo diario [5].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [8].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Atzori, L., Iera, A., & Morabito, G. (2010). The Internet of Things: A survey. *Computer Networks, 54*(15), 2787–2805. https://doi.org/10.1016/j.comnet.2010.05.010

**[4]** Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3.ª ed.). OTexts. https://otexts.com/fpp3/

**[5]** Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M. (2015). *Time Series Analysis: Forecasting and Control* (5.ª ed.). Wiley.

**[6]** Seabold, S., & Perktold, J. (2010). statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92–96. https://doi.org/10.25080/Majora-92bf1922-011

**[7]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[8]** Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55